# Iceberg Housekeeping with Spark

## 1. Initializing the Spark Session
In this lab, we will be using **Apache Spark** as our compute engine to interact with **Apache Iceberg**. Every Iceberg operation requires a Spark Session that is configured to talk to the Iceberg Catalog.

![image1.jpeg](assets/image1.jpeg)
    
**Directions:**
* Enter your assigned **username** (e.g., `user001`) to ensure you are working in your own isolated database.
* Execute the cell to start your Spark Application.## 1. Initializing the Spark Session
In this lab, we will be using **Apache Spark** as our compute engine to interact with **Apache Iceberg**. Every Iceberg operation requires a Spark Session that is configured to talk to the Iceberg Catalog.

**Directions:**
* Enter your assigned **username** (e.g., `user001`) to ensure you are working in your own isolated database.
* Execute the cell to start your Spark Application.

In [ ]:
# Enter your assigned user below
username = "user003"

from datetime import datetime
import cml.data_v1 as cmldata

SPARK_CONNECTION_NAME = "hol-aw-dl"
conn_spark = cmldata.get_connection(SPARK_CONNECTION_NAME)
spark = conn_spark.get_spark_session()

print("Code block completed")

## 2. Examining Table Metadata
Before we optimize the table, we need to understand its current structure. We will check the column types, the underlying storage format, and the table properties.

![image2.jpeg](assets/image2.jpeg)

    
**Directions:**
* Review the `DESCRIBE` output to see the schema.
* Review the `SHOW CREATE TABLE` output. Look for the `LOCATION` to see where the data is stored in S3.
* Check the `TBLPROPERTIES` to see if the table is already using the Iceberg format.

In [ ]:
# Describe the table
spark.sql("DESCRIBE lakehouse_optimizer_{}.sensor_telemetry".format(username)).show()

# Show the table's CREATE statement
spark.sql("SHOW CREATE TABLE lakehouse_optimizer_{}.sensor_telemetry".format(username)).show(truncate=False)

# Show the table properties
spark.sql("SHOW TBLPROPERTIES lakehouse_optimizer_{}.sensor_telemetry".format(username)).show(truncate=False)

print("Code block completed")

## 3. Baseline Statistics: Identifying "Small File Syndrome"
A common performance killer in Data Lakes is having thousands of tiny files. This creates a massive overhead for the Spark planner. We will run a custom utility script to see exactly how many files make up your table and what their average size is.

![image3.jpeg](assets/image3.jpeg)


**Directions:**
* Note the **Total Data Files** count.
* Note the **Average Data File Size**. If it is under 10MB, the table is a prime candidate for compaction.

In [ ]:
%run -i assets/get_table_stats.py lakehouse_optimizer_$username sensor_telemetry

print("Code block completed")

## 4. Compaction: Optimizing the Data Layer
Iceberg provides a powerful stored procedure called `rewrite_data_files`. This doesn't change the data itself; instead, it reads the 900+ tiny files and merges them into a few large, healthy files (targeting ~128MB). 

![image4.jpeg](assets/image4.jpeg)

**Directions:**
* We are setting the `target-file-size-bytes` to **128MB** (`134217728`).
* Observe the output; it will tell you how many files were rewritten and how many new, larger files were created.

In [ ]:
# Rewrite small files to 128MB
spark.sql(f"""
    CALL system.rewrite_data_files(
        table => 'lakehouse_optimizer_{username}.sensor_telemetry', 
        options => map('target-file-size-bytes','134217728') 
    )
""").show()

print("Code block completed")

## 5. Post-Compaction Audit
Let's see the physical impact of the compaction command. We expect to see the **Total Files** for the current snapshot to drop significantly and the **Average File Size** to increase.

![image5.jpeg](assets/image5.jpeg)

**Directions:**
* Run the stats script again.
* **Observe:** You will likely see "total data" (Physical S3) is still high, while "data (current snapshot)" (Logical Table) is now low. This is because Iceberg keeps history for Time Travel!

In [ ]:
%run -i assets/get_table_stats.py lakehouse_optimizer_$username sensor_telemetry

print("Code block completed")

## 6.1 Benchmarking: Small Files vs. Optimized Files
Now for the "Proof of Concept." We have an exact duplicate of our table prior to running compaction. Let's run the same analytical query against our **Original** (Bloated) table and our **Optimized** (Compacted) table. This will give us an understanding of how compaction can affected query performance.

![image6.jpeg](assets/image6.jpeg)

Firstly let's get the table stats for our original table.
**Directions:**
* Run the stats script again but this time we'll look at a copy of the table called sensor_telemetry_original.
* **Observe:** You will likely see "total data" (Physical S3) and "data (current snapshot)" (Logical Table) is the same. No compaction has been run on this table.

In [ ]:
%run -i assets/get_table_stats.py lakehouse_optimizer_$username sensor_telemetry_original

print("Code block completed")

Now let's run the queries below. We're comparing two tables that we exactly the same to start with, however we've run compaction on one of them.
The goal here is to see how much of a diiference compaction can make with query performance.


**Directions:**
* We use `spark.catalog.clearCache()` to ensure a fair "cold start" test.
* Compare the **Execution Time**. You should see a significant speedup on the optimized table because Spark spends less time "opening" files.

In [ ]:
tables = {
    "Bloated Table (Small Files)": f"lakehouse_optimizer_{username}.sensor_telemetry_original",
    "Optimized Table (Compacted)": f"lakehouse_optimizer_{username}.sensor_telemetry"
}

spark.sparkContext.setLogLevel("ERROR")

# Dictionary to store results for comparison
results = {}

for label, table in tables.items():
    print(f"Testing performance for: {label}...")
    
    spark.catalog.clearCache()
    
    start_time = time.time()
    
    df = spark.sql(f"""
        SELECT status, count(*), avg(temperature) 
        FROM {table} 
        GROUP BY status
    """)
    df.collect()
    
    duration = time.time() - start_time
    results[label] = duration
    print(f"--- Execution Time: {duration:.2f} seconds ---\n")

print("Code block completed")

## 7. Storage Reclamation: Expiring Old Snapshots
Even though our table is now fast, the old "trash" files are still in S3 taking up space. To save money and clean up, we must **Expire Snapshots**. This removes the metadata links to the old versions of the data. Be careful though because expiring snapshots prevents you from being able to time-travel back to the table state at a point in time. You may want to be more selective in which snapshots you expire to still alow time-travel but the principle is the same.

![image7.jpeg](assets/image7.jpeg)

**Directions:**
* We will keep only the **last 1 snapshot**.
* Once this is done, the "history" is gone, the old files are physically deleted. 

In [ ]:
%run -i assets/get_table_stats.py lakehouse_optimizer_$username sensor_telemetry

print("Code block completed")

In [ ]:
# 1. Get the current time as a formatted string in Python
now = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

# 2. Show all Snapshots 
snapshots_df = spark.sql("""SELECT * FROM lakehouse_optimizer_{}.sensor_telemetry.snapshots""".format(username))
print("SNAPSHOTS BEFORE EXPIRY")
snapshots_df.show()

print("EXPIRE ALL SNAPSHOPS EXCEPT THE LATEST")
# Expire older snaphots
spark.sql(f"""
    CALL system.expire_snapshots(
        table => 'lakehouse_optimizer_{username}.sensor_telemetry',
        older_than => TIMESTAMP '{now}',
        retain_last => 1
    )
""").show()

# 2. Show all Snapshots 
snapshots_df = spark.sql("""SELECT * FROM lakehouse_optimizer_{}.sensor_telemetry.snapshots""".format(username))
print("SNAPSHOTS AFTER EXPIRY")
snapshots_df.show()

print("Code block completed")

## 8. Final Verification
This is your "Mission Accomplished" check. After expiring snapshots and removing orphans, your **Total Data** in S3 should finally match your **Current Snapshot**.

![image8.jpeg](assets/image8.jpeg)

**Directions:**
* Run the final stats check.
* Verify that the physical S3 file count is now the same as the logical Iceberg file count.

In [ ]:
%run -i assets/get_table_stats.py lakehouse_optimizer_$username sensor_telemetry

print("Code block completed")

## &#x1F600; Well done - You've completed this section of the labs!